# JAX array arithmetic and original CPU validation

This notebook isolates arithmetic on identical deterministic inputs. Each example compares ordinary PyCBC CPU execution, default JAX execution and a JAX scheme that routes **only the selected operation** to the original CPU helper. Native validation is optional; the default remains on device.

Assertions check the native result against the CPU result exactly, including dtype, shape and bytes. Default discrepancies are measured, not assumed. These examples establish array-operation behavior in the displayed environment; they do not establish complete-search equivalence. Install PyCBC, JAX and Jupyter in the same environment before running.

In [1]:
import math
import platform
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import numpy as np
import scipy
import jax
import jaxlib
import pycbc
from pycbc.scheme import CPUScheme, JAXScheme
from pycbc.types import Array

DEVICE = "cpu"  # Change to "cuda:0" to repeat on an available NVIDIA GPU.
with JAXScheme(DEVICE) as context:
    metadata = {
        "Python": platform.python_version(),
        "CPU architecture": platform.machine(),
        "PyCBC": pycbc.__version__,
        "NumPy": np.__version__,
        "SciPy": scipy.__version__,
        "JAX": jax.__version__,
        "jaxlib": jaxlib.__version__,
        "device platform": context.jax_device.platform,
        "device kind": context.jax_device.device_kind,
        "JAX x64": jax.config.jax_enable_x64,
    }
assert metadata["JAX x64"], "These examples require JAX 64-bit support."
for name, value in metadata.items():
    print(f"{name}: {value}")
np.set_printoptions(precision=17)


Python: 3.12.7
CPU architecture: arm64
PyCBC: 0.0a9304
NumPy: 2.5.2
SciPy: 1.16.3
JAX: 0.11.1
jaxlib: 0.11.1
device platform: cpu
device kind: cpu
JAX x64: True


## Identical inputs and exact checks

Fresh arrays prevent one backend's mutation from becoming another backend's input. Results are copied to host storage before leaving each context. CPU runs use `CPUScheme`; native runs select `reference_operations=(operation,)` inside `JAXScheme`. No other operation is switched.

In [2]:
def snapshot(value):
    if isinstance(value, tuple):
        return tuple(snapshot(part) for part in value)
    if isinstance(value, Array):
        return np.array(value.numpy(), copy=True)
    return np.array(value, copy=True)


def run(operation, values, *, other=None, weight=None, factor=None,
        cpu=False, reference=()):
    context = CPUScheme() if cpu else JAXScheme(
        DEVICE, reference_operations=reference)
    with context:
        a = Array(values.copy())
        b = a if other is None else Array(other.copy())
        if operation in ("inner", "dot"):
            result = getattr(a, operation)(b)
        elif operation == "weighted_inner":
            result = a.weighted_inner(b, Array(weight.copy()))
        elif operation == "multiply_and_add":
            result = a.multiply_and_add(b, factor)
        else:
            result = getattr(a, operation)()
        return snapshot(result)


def assert_exact(actual, expected):
    if isinstance(expected, tuple):
        assert isinstance(actual, tuple) and len(actual) == len(expected)
        for a, b in zip(actual, expected):
            assert_exact(a, b)
    else:
        assert actual.shape == expected.shape
        assert actual.dtype == expected.dtype
        assert actual.tobytes() == expected.tobytes()


results = {}

def compare(name, operation, values, **kwargs):
    cpu = run(operation, values, cpu=True, **kwargs)
    default = run(operation, values, **kwargs)
    native = run(operation, values, reference=(operation,), **kwargs)
    assert_exact(native, cpu)
    results[name] = (cpu, default, native)
    print(f"{name}: input dtype={values.dtype}, size={values.size}")
    print("  CPU:", cpu)
    print("  default JAX:", default)
    if not isinstance(cpu, tuple):
        print("  maximum absolute difference:", np.max(np.abs(default - cpu)))
    print("  native route: exact dtype/shape/byte check passed")
    return cpu, default, native


## Product precision: real and complex inner products

The CPU forms single-precision products before double-precision accumulation. JAX widens the operands before multiplication. For a one-element real self-inner product, there is no reduction-order ambiguity: the following controls isolate the product-rounding difference.

The complex self-inner specialization in JAX sums squared real and imaginary parts. The CPU performs complex conjugate multiplication, so its intermediate single-precision arithmetic can also leave a small imaginary rounding residual. The notebook reports the actual result rather than discarding that component.

In [3]:
real = np.array([1.234567], dtype=np.float32)
cpu, default, native = compare("real self-inner", "inner", real)
rounded_product = np.float64(np.float32(real[0] * real[0]))
widened_product = np.float64(real[0]) * np.float64(real[0])
assert_exact(cpu, np.asarray(rounded_product))
assert_exact(default, np.asarray(widened_product))
assert rounded_product != widened_product
print("  float32 product then float64:", rounded_product)
print("  float64 operands then product:", widened_product)

complex_values = np.array([1.234567 + 2.345678j], dtype=np.complex64)
compare("complex self-inner", "inner", complex_values)
compare("real inner", "inner", real,
        other=np.array([2.345678], dtype=np.float32))
compare("complex inner", "inner", complex_values,
        other=np.array([2.345678 - 1.234567j], dtype=np.complex64))


real self-inner: input dtype=float32, size=1
  CPU: 1.5241557359695435
  default JAX: 1.5241557914777246
  maximum absolute difference: 5.550818116262235e-08
  native route: exact dtype/shape/byte check passed
  float32 product then float64: 1.5241557359695435
  float64 operands then product: 1.5241557914777246
complex self-inner: input dtype=complex64, size=1
  CPU: (7.026361465454102-3.9499440163126565e-08j)
  default JAX: (7.026361498305917+0j)
  maximum absolute difference: 5.137555405249631e-08
  native route: exact dtype/shape/byte check passed
real inner: input dtype=float32, size=1
  CPU: 2.8958969116210938
  default JAX: 2.8958968721216536
  maximum absolute difference: 3.9499440163126565e-08
  native route: exact dtype/shape/byte check passed
complex inner: input dtype=complex64, size=1
  CPU: (-3.9499440163126565e-08-7.026361465454102j)
  default JAX: -7.026361498305917j
  maximum absolute difference: 5.137555405249631e-08
  native route: exact dtype/shape/byte check passed


(array(-3.9499440163126565e-08-7.026361465454102j),
 array(0.-7.026361498305917j),
 array(-3.9499440163126565e-08-7.026361465454102j))

## Weighted inner products

The CPU product and division occur in the input precision before its double-precision sum. JAX widens the operands first. These examples keep the weights and both vector inputs fixed; the native switch restores the original product, division and accumulation together.

In [4]:
weights = np.array([1.111111], dtype=np.float32)
compare("real weighted inner", "weighted_inner", real, weight=weights)
compare("complex weighted inner", "weighted_inner", complex_values,
        weight=weights)


real weighted inner: input dtype=float32, size=1
  CPU: 1.3717402219772339
  default JAX: 1.3717402940920451
  maximum absolute difference: 7.211481123370334e-08
  native route: exact dtype/shape/byte check passed
complex weighted inner: input dtype=complex64, size=1
  CPU: (6.323725700378418-3.554949756789938e-08j)
  default JAX: (6.32372572539875+0j)
  maximum absolute difference: 4.3471643796460404e-08
  native route: exact dtype/shape/byte check passed


(array(6.323725700378418-3.554949756789938e-08j),
 array(6.32372572539875+0.j),
 array(6.323725700378418-3.554949756789938e-08j))

## Reduction order and cancellation

Addition is not associative in floating-point arithmetic: `(1e16 + -1e16) + 1` retains the small term, while `1e16 + (-1e16 + 1)` loses it. NumPy and JAX reductions or scans can group contributions differently. Shape, device and library versions can change the grouping.

The patterns below amplify that sensitivity. `math.fsum` supplies a diagnostic sum; it does not replace the original CPU validation target. The assertions require the selected native route to match CPU, without requiring a default discrepancy on every platform.

In [5]:
large = np.float64(1e16)
print("two addition orders:", (large + -large) + 1, large + (-large + 1))
sum_values = np.tile(np.array([1e16, 1, -1e16, 1], dtype=np.float64), 2)
compare("cancellation sum", "sum", sum_values)
print("  math.fsum diagnostic:", math.fsum(sum_values))

dot_values = np.tile(np.array([1e8, 1, -1e8, 1], dtype=np.float32), 16)
compare("cancellation dot", "dot", dot_values,
        other=np.ones_like(dot_values))

scan_values = np.tile(np.array([1e8, 1, -1e8, 1], dtype=np.float32), 8)
compare("cancellation cumsum", "cumsum", scan_values)


two addition orders: 1.0 0.0
cancellation sum: input dtype=float64, size=8
  CPU: 0.0
  default JAX: 1.0
  maximum absolute difference: 1.0
  native route: exact dtype/shape/byte check passed
  math.fsum diagnostic: 4.0
cancellation dot: input dtype=float32, size=64
  CPU: 0.0
  default JAX: 2.0
  maximum absolute difference: 2.0
  native route: exact dtype/shape/byte check passed
cancellation cumsum: input dtype=float32, size=32
  CPU: [1.e+08 1.e+08 0.e+00 1.e+00 1.e+08 1.e+08 0.e+00 1.e+00 1.e+08 1.e+08
 0.e+00 1.e+00 1.e+08 1.e+08 0.e+00 1.e+00 1.e+08 1.e+08 0.e+00 1.e+00
 1.e+08 1.e+08 0.e+00 1.e+00 1.e+08 1.e+08 0.e+00 1.e+00 1.e+08 1.e+08
 0.e+00 1.e+00]
  default JAX: [1.e+08 1.e+08 0.e+00 1.e+00 1.e+08 1.e+08 0.e+00 1.e+00 1.e+08 1.e+08
 0.e+00 1.e+00 1.e+08 1.e+08 0.e+00 1.e+00 1.e+08 1.e+08 1.e+00 2.e+00
 1.e+08 1.e+08 1.e+00 2.e+00 1.e+08 1.e+08 1.e+00 2.e+00 1.e+08 1.e+08
 1.e+00 2.e+00]
  maximum absolute difference: 1.0
  native route: exact dtype/shape/byte check passed

(array([1.e+08, 1.e+08, 0.e+00, 1.e+00, 1.e+08, 1.e+08, 0.e+00, 1.e+00,
        1.e+08, 1.e+08, 0.e+00, 1.e+00, 1.e+08, 1.e+08, 0.e+00, 1.e+00,
        1.e+08, 1.e+08, 0.e+00, 1.e+00, 1.e+08, 1.e+08, 0.e+00, 1.e+00,
        1.e+08, 1.e+08, 0.e+00, 1.e+00, 1.e+08, 1.e+08, 0.e+00, 1.e+00],
       dtype=float32),
 array([1.e+08, 1.e+08, 0.e+00, 1.e+00, 1.e+08, 1.e+08, 0.e+00, 1.e+00,
        1.e+08, 1.e+08, 0.e+00, 1.e+00, 1.e+08, 1.e+08, 0.e+00, 1.e+00,
        1.e+08, 1.e+08, 1.e+00, 2.e+00, 1.e+08, 1.e+08, 1.e+00, 2.e+00,
        1.e+08, 1.e+08, 1.e+00, 2.e+00, 1.e+08, 1.e+08, 1.e+00, 2.e+00],
       dtype=float32),
 array([1.e+08, 1.e+08, 0.e+00, 1.e+00, 1.e+08, 1.e+08, 0.e+00, 1.e+00,
        1.e+08, 1.e+08, 0.e+00, 1.e+00, 1.e+08, 1.e+08, 0.e+00, 1.e+00,
        1.e+08, 1.e+08, 0.e+00, 1.e+00, 1.e+08, 1.e+08, 0.e+00, 1.e+00,
        1.e+08, 1.e+08, 0.e+00, 1.e+00, 1.e+08, 1.e+08, 0.e+00, 1.e+00],
       dtype=float32))

## BLAS multiply and add

CPU `multiply_and_add` calls BLAS AXPY; its multiply and add can be fused. The default JAX array implementation evaluates separate array operations. With the inputs below, two float32 roundings give zero; one final rounding retains `-2**-46`.

The controls compute both sequences exactly for these representable inputs. Backend fusion behavior depends on the build and device, so the output identifies which control each backend matches. The selected native route must match the installed CPU implementation.

In [6]:
factor = np.float32(1 + 2**-23)
other = np.array([1 - 2**-23], dtype=np.float32)
destination = np.array([-1], dtype=np.float32)
cpu, default, native = compare("multiply and add", "multiply_and_add",
                               destination, other=other, factor=factor)
separate = np.float32(destination[0] + np.float32(factor * other[0]))
one_rounding = np.float32(np.float64(destination[0])
                          + np.float64(factor) * np.float64(other[0]))
assert separate == 0
assert one_rounding == np.float32(-2**-46)
print("  separate-rounding control:", separate)
print("  one-rounding control:", one_rounding)
for name, result in (("CPU", cpu), ("default JAX", default)):
    print(f"  {name} matches separate rounding:", result[0] == separate,
          "; matches one rounding:", result[0] == one_rounding)


multiply and add: input dtype=float32, size=1
  CPU: [-1.4210855e-14]
  default JAX: [0.]
  maximum absolute difference: 1.4210855e-14
  native route: exact dtype/shape/byte check passed
  separate-rounding control: 0.0
  one-rounding control: -1.4210855e-14
  CPU matches separate rounding: False ; matches one rounding: True
  default JAX matches separate rounding: True ; matches one rounding: False


## Other supported native routes

Finite-input examples below check `squared_norm`, `min`, `max`, `max_loc`, `abs_max_loc` and `abs_arg_max`. An exact native result does not imply that a default discrepancy exists for every input. Overflow and nonfinite values can depend on CPU compiler flags and backend implementation; these finite controls make no claim about them.

In [7]:
finite_real = np.array([-3, 2, 1], dtype=np.float32)
for operation in ("min", "max", "max_loc", "abs_max_loc", "abs_arg_max"):
    compare(operation, operation, finite_real)
compare("squared norm", "squared_norm", complex_values)
assert len(results) == 16
print(f"Checked {len(results)} comparisons; every selected native result matched CPU exactly.")


min: input dtype=float32, size=3
  CPU: -3.0
  default JAX: -3.0
  maximum absolute difference: 0.0
  native route: exact dtype/shape/byte check passed
max: input dtype=float32, size=3
  CPU: 2.0
  default JAX: 2.0
  maximum absolute difference: 0.0
  native route: exact dtype/shape/byte check passed
max_loc: input dtype=float32, size=3
  CPU: (array(2., dtype=float32), array(1))
  default JAX: (array(2.), array(1))
  native route: exact dtype/shape/byte check passed
abs_max_loc: input dtype=float32, size=3
  CPU: (array(3., dtype=float32), array(0))
  default JAX: (array(3.), array(0))
  native route: exact dtype/shape/byte check passed
abs_arg_max: input dtype=float32, size=3
  CPU: 0
  default JAX: 0
  maximum absolute difference: 0
  native route: exact dtype/shape/byte check passed
squared norm: input dtype=complex64, size=1
  CPU: [7.0263615]
  default JAX: [7.0263615]
  maximum absolute difference: 0.0
  native route: exact dtype/shape/byte check passed
Checked 16 comparisons; e

## Using these controls in a larger calculation

Keep identical inputs at the operation boundary, enable only the operation being isolated, and compare its output before feeding it into the remaining JAX calculation. Combine operation names when more than one boundary needs the original calculation, for example `JAXScheme(DEVICE, reference_operations=("inner", "cumsum"))`.

Native routes perform host conversion and invoke the original CPU helper. They can be much slower, do not execute inside JAX tracing and cannot be differentiated through that boundary. The controls select named PyCBC array operations; other JAX kernels need their own stage controls. Exact equality here uses the same CPU build and input bytes, and does not promise equality across different BLAS builds or complete searches.

See [JAX array numerical differences](../../docs/jax_numerical_differences.rst) and [JAX's numerical FAQ](https://docs.jax.dev/en/latest/faq.html#jit-changes-the-exact-numerics-of-outputs).